# Инференс: запрос → рекомендации

Пользователь (история прослушиваний + профиль) пишет запрос (на английском, как в датасете), в ответ — текст и 20 треков.
Вся логика в пакете `recsys/`, здесь только запуск.

* `experiments.ipynb` — почему такая выдача: разбор по шагам, сравнение вариантов, LLM
* `scripts/evaluate.py` — метрики на валидации
* `examples/candgen.ipynb` — проверка сервиса HNSW руками

## 1. Настройки

По умолчанию — настоящий датасет и HNSW с сервера (остальные источники считаются здесь):

* Источники кандидатов: `relisten` (своя история), `audio` (MuQ), `bm25` (локальный, индекс карточек — `python scripts/make_index.py`)
  и `hnsw` (сервис на сервере). `USE_CANDGEN = True` — HNSW включён; адрес и ключ — `HNSW_URL`, `HNSW_API_KEY`
  (файл `.env` в корне или секреты проекта DataSphere). `False` — без него.
* `DATA = "crs"` — датасет из `DATA_DIR` (`docs/dataset.md`);
  `"synthetic"` — сгенерированные данные, работает без файлов (только чтобы проверить код).
* `RANKER` — шаг 4: `"stub"` — порядок после слияния (RRF), `"heuristic"` — взвешенная сумма признаков
  (`recsys/ranking.py`). LightGBM-ранкер из ветки `dev-ranker` появится здесь после её вливания.
* `USE_LLM_SERVICE = True` — текст ответа пишет отдельный Gemma-сервис на той же GPU-ВМ
  (`configs/gemma.yaml`). Сначала настройте его через `llm_gemma_service/gemma_datasphere_service.ipynb`.
  Уже загруженная модель используется повторно при обновлении pipeline. `False` — шаблонный ответ.
* HNSW: вектор запроса строим сами (EmbeddingGemma-2) — на GPU, если он есть (DataSphere), иначе на CPU.
  Один раз на машину: `python scripts/make_embed.py` (пакеты, веса, проверка на сервере). Без модели HNSW пропускается.

In [ ]:
USE_CANDGEN = True         # HNSW с сервера; False — без него (relisten, audio, bm25)
DATA = "crs"               # настоящий датасет; "synthetic" — сгенерированные примеры, только чтобы проверить код
DATA_DIR = "music4all_crs"  # tracks_meta, test_public, test_public_queries, test_public_qrels
N_USERS = 300              # сколько пользователей загрузить (None — все, долго)
RANKER = "stub"            # "stub" — порядок RRF, "heuristic" — взвешенная сумма признаков
USE_LLM_SERVICE = False    # True — описание выдачи пишет Gemma-сервис DataSphere (llm.gemma_service); локально нет

# адреса и ключи сервисов — из файла .env или секретов DataSphere; можно и здесь:
# os.environ["HNSW_URL"] = "http://<IP>:8001"; os.environ["HNSW_API_KEY"] = "..."

In [ ]:
%pip install -q -r requirements.txt

## 2. Каталог и пайплайн

Загружается один раз. Для HNSW печатается, доступен ли сервис и какая версия индекса.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace

from recsys.config import load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Message

files = ["configs/default.yaml"]
overrides = {"data": {"source": DATA, "crs": {"dir": DATA_DIR, "n_users": N_USERS}}, "ranker": {"type": RANKER}}
if not USE_CANDGEN or DATA == "synthetic":  # сервисы знают только каталог датасета
    overrides["retrieval"] = {"hnsw": {"enabled": False}}
if USE_LLM_SERVICE:
    files.append("configs/gemma.yaml")
cfg = load_config(files, overrides)
data = load_data(cfg)
pipe = Pipeline.from_config(cfg, data.catalog)

for source in pipe.retrievers:
    if getattr(source, "remote", False):
        print(source.name, source.health())

## 3. Запрос

Пользователь из `test_public`: его история, профиль и настоящий запрос; известен трек, который он на самом деле
послушал (цель). `REQUEST` — номер запроса; `TEXT` — свой текст вместо запроса из датасета (история та же).

In [ ]:
REQUEST = 0      # номер запроса в test_public (загружено N_USERS пользователей, до 5 запросов у каждого)
TEXT = None      # None — настоящий запрос из датасета (известен трек, который послушали); или свой текст:
# TEXT = "something calm for the evening, no rap"

user = data.requests[REQUEST]
# запрос нужного типа: user = next(r for r in data.requests if r.meta["query_type"] == "similar_to")
request = replace(user, dialog=[Message("user", TEXT)]) if TEXT else user
print("запрос:", request.dialog[-1].text)
if not TEXT:
    print("тип:", request.meta["query_type"], "| цель:", request.meta["artist"], "—", request.meta["title"],
          "| уже слушал:", not request.meta["is_new"])
print("треков в истории:", len(request.history))
print("профиль:", request.user_info[:200], "...")

## 4. Ответ

In [ ]:
response = pipe.run(request, debug=True)
print(response.text)
response.to_frame()

Если запрос из датасета: на каком месте трек, который пользователь послушал.

In [ ]:
if not TEXT:
    target = request.target_ids[0]
    place = response.track_ids.index(target) + 1 if target in response.track_ids else "нет в top-20"
    print("цель:", request.meta["artist"], "—", request.meta["title"], "| место в выдаче:", place)

## 5. Почему такая выдача

Как понят запрос (английские теги — именно они уходят в BM25) и сколько кандидатов дал каждый источник.
`relisten` — треки из своей истории, `audio` — похожие по звучанию (MuQ), `bm25` — локальный BM25 по карточке трека,
`hnsw` — сервис на сервере.

In [ ]:
print("понял как:", response.summary.text)
print("теги:", response.summary.include_tags, "| исключить:", response.summary.exclude_tags)
print("кандидатов:", response.debug["n_candidates"])
print("ранкер:", type(pipe.ranker).__name__)
hnsw = next((r for r in pipe.retrievers if r.name == "hnsw"), None)
if hnsw is not None:
    print("hnsw: вектор на", hnsw.embedder.device, "| ошибка:", hnsw.embed_error)

## 6. Что вернули BM25 и HNSW

Первые кандидаты BM25 (локальный) и HNSW (сервер) с местом и оценкой. Цель отмечена `<<`. Если BM25 пуст — смотрите,
какие слова ушли в него (раздел 5, «теги»); если пуст HNSW — строку «hnsw: вектор на …» в разделе 5.

In [ ]:
import pandas as pd

target = request.target_ids[0] if not TEXT else None
for name in ["bm25", "hnsw"]:
    cands = response.debug["candidates"].get(name, [])
    print(f"{name}: {len(cands)} кандидатов")
    rows = [{"место": c.rank, "оценка": round(c.score, 3), "артист": data.catalog.row(c.track_id)["artist"],
             "трек": data.catalog.row(c.track_id)["title"], "": "<<" if c.track_id == target else ""}
            for c in cands[:10]]
    if rows:
        display(pd.DataFrame(rows))

## 7. Следующая реплика

Диалог продолжается: уже показанное не повторяется, новые пожелания перекрывают старые.

In [ ]:
request2 = request.next_turn(response, "now something more energetic")
response2 = pipe.run(request2)
print(response2.text)
response2.to_frame()